# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# !cp -r "/content/drive/My Drive/pixels-to-predictions/" "/content"


Mounted at /content/drive


In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader



# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("/content/drive/My Drive/pixels-to-predictions/")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Load Data

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(len(train_df) + len(val_df) + len(test_df))
train_df.head(2)
# 3q

Train: 3,109 | Val: 1,048 | Test: 1,008
5165


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


### Taking a look at the data

In [ ]:
print(f"\nDataset shape: {train_df.shape}")
print(f"\nData types:")
print(train_df.dtypes)
print(f"\nBasic statistics:")
print(train_df.describe())
print(f"\nMissing values:")
print(train_df.isnull().sum())


Dataset shape: (3109, 15)

Data types:
id             object
image_path     object
question       object
choices        object
num_choices     int64
answer          int64
hint           object
lecture        object
solution       object
task           object
grade          object
subject        object
topic          object
category       object
skill          object
dtype: object

Basic statistics:
       num_choices       answer
count  3109.000000  3109.000000
mean      3.109038     1.022194
std       0.771524     0.952452
min       2.000000     0.000000
25%       3.000000     0.000000
50%       3.000000     1.000000
75%       4.000000     2.000000
max       5.000000     4.000000

Missing values:
id               0
image_path       0
question         0
choices          0
num_choices      0
answer           0
hint           724
lecture        440
solution       529
task             0
grade            0
subject          0
topic            0
category         0
skill            0
dtype: 

### Cleaning up data

In [ ]:
# Fill missing text fields with empty strings
train_df['hint'] = train_df['hint'].fillna('')
train_df['lecture'] = train_df['lecture'].fillna('')
train_df['solution'] = train_df['solution'].fillna('')

# Now verify:
print(train_df.isnull().sum())

id             0
image_path     0
question       0
choices        0
num_choices    0
answer         0
hint           0
lecture        0
solution       0
task           0
grade          0
subject        0
topic          0
category       0
skill          0
dtype: int64


In [ ]:
val_set = {"images/val/" + i.name for i in (DATA_DIR / "images/images/val").iterdir()}
print(val_set)
print("images/val/val_00671.png" in val_set)
print(val_df["image_path"])
print("Before:", len(val_df))
val_df = val_df[val_df["image_path"].isin(val_set)]
print("After:", len(val_df))


{'images/val/val_02203.png', 'images/val/val_03216.png', 'images/val/val_01607.png', 'images/val/val_01881.png', 'images/val/val_00509.png', 'images/val/val_01125.png', 'images/val/val_01639.png', 'images/val/val_01734.png', 'images/val/val_03827.png', 'images/val/val_01249.png', 'images/val/val_03558.png', 'images/val/val_02222.png', 'images/val/val_03997.png', 'images/val/val_00226.png', 'images/val/val_00652.png', 'images/val/val_03590.png', 'images/val/val_00870.png', 'images/val/val_01137.png', 'images/val/val_03363.png', 'images/val/val_02901.png', 'images/val/val_02084.png', 'images/val/val_00754.png', 'images/val/val_03910.png', 'images/val/val_03618.png', 'images/val/val_02831.png', 'images/val/val_02408.png', 'images/val/val_03334.png', 'images/val/val_00836.png', 'images/val/val_01106.png', 'images/val/val_00100.png', 'images/val/val_01039.png', 'images/val/val_00017.png', 'images/val/val_01906.png', 'images/val/val_02778.png', 'images/val/val_03585.png', 'images/val/val_026

### Encoding Data

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Performing Ordinal Encoding for the grade level
grade_levels = [f"grade{i}" for i in range(1,13)]
print(grade_levels)

ord_enc = OrdinalEncoder(categories=[grade_levels])
train_df["grade_encoded"] = ord_enc.fit_transform(train_df[['grade']])
train_df.drop(columns=["grade"])

val_df["grade_encoded"] = ord_enc.fit_transform(val_df[['grade']])
val_df.drop(columns=["grade"])

test_df["grade_encoded"] = ord_enc.fit_transform(test_df[['grade']])
test_df.drop(columns=["grade"])
print(train_df["grade_encoded"].head())


['grade1', 'grade2', 'grade3', 'grade4', 'grade5', 'grade6', 'grade7', 'grade8', 'grade9', 'grade10', 'grade11', 'grade12']
0    7.0
1    7.0
2    7.0
3    7.0
4    7.0
Name: grade_encoded, dtype: float64


# Data Augmentation:
Use the results of the EDA to augment data

## Shuffling answer choices:
By shuffling the where the correct answer is, we can prevent the model from memorizing

In [ ]:
# import random

# def get_shuffled_choices(choices, correct_answer_idx):
#     """
#     Shuffles a list of choices and returns the new list along with the
#     updated index of the correct answer.

#     Args:
#         choices (list): List of strings (the multiple choice options).
#         correct_answer_idx (int): The index (0-based) of the correct answer.

#     Returns:
#         tuple: (shuffled_choices_list, new_correct_answer_idx)
#     """
#     # Create a list of tuples: (choice_text, original_index)
#     indexed_choices = list(enumerate(choices))

#     # Avoid shuffling
#     is_true_false = False
#     for original_idx, choice_text in indexed_choices:
#       if str(choice_text).lower().strip() in ["true", "false", "yes", "no"]:
#         is_true_false = True
#         break

#     # Shuffle the tuples
#     if (not is_true_false):
#       random.shuffle(indexed_choices)

#     # Extract the shuffled text and find where the original correct index moved to
#     shuffled_choices = [choice_text for original_idx, choice_text in indexed_choices]
#     new_correct_idx = next(i for i, (original_idx, _) in enumerate(indexed_choices)
#                            if original_idx == correct_answer_idx)

#     return shuffled_choices, new_correct_idx


# # --- TEST SUITE ---
# test_cases = [
#     {
#         "name": "Standard Multiple Choice (Should Shuffle)",
#         "choices": ["Gravity", "Inertia", "Friction", "Magnetism"],
#         "idx": 0
#     },
#     {
#         "name": "True/False (Should NOT Shuffle)",
#         "choices": ["True", "False"],
#         "idx": 0
#     },
#     {
#         "name": "Yes/No (Should NOT Shuffle)",
#         "choices": ["Yes", "No"],
#         "idx": 1
#     },
#     {
#         "name": "2-Choice Science (Should NOT Shuffle per your len > 2 rule)",
#         "choices": ["Mitosis", "Meiosis"],
#         "idx": 0
#     }
# ]

# for case in test_cases:
#     shuffled, new_idx = get_shuffled_choices(case["choices"], case["idx"])
#     print(f"--- {case['name']} ---")
#     print(f"Original: {case['choices']} (Correct: {case['choices'][case['idx']]})")
#     print(f"Shuffled: {shuffled} (New Correct: {shuffled[new_idx]})")
#     print(f"Status: {'STAYED FIXED' if shuffled == case['choices'] else 'SHUFFLED'}\n")

## Augmenting Images

In [ ]:
from torchvision import transforms

vision_aug = transforms.Compose([
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    # transforms.RandomRotation(degrees=5), # Rotates randomly between -5 and +5 degrees
])

# Prompt Engineering

In [ ]:
# ── 2b. Prompt Engineering ───────────────────────────────────────────────────
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    # 1. Inject the 'Expertise' Metadata Header
    # This acts as a trigger for the model's pre-trained science knowledge
    subject = str(row.get("subject", "Science")).title()
    topic   = str(row.get("topic", "General")).title()
    grade   = "Grade " + str(int(row.get("grade_encoded", "")))

    header = f"Task: {subject} | Category: {topic} | Level: {grade}\n"

    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )


    prompt = header
    # prompt += "<image>\n"

    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f" {CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))

Task: Natural Science | Category: Literacy-In-Science | Level: Grade 7
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. An

# Balancing Data

### Using a multi-stage stratified strategy

In [ ]:
#  train_df_final = train_df.sample(frac=)

In [ ]:
#Joint strategy
import pandas as pd

def create_static_stratified_df(df, weight_cols, total_samples=None):
    """
    Uses the weighting logic to generate a static, balanced DataFrame.
    """
    df = df.copy()
    # 1. Create the composite key
    df['_strat_key'] = df[weight_cols].astype(str).agg('__'.join, axis=1)

    # 2. Calculate Weights
    strat_counts = df['_strat_key'].value_counts()
    weights = df['_strat_key'].map(lambda k: 1.0 / np.sqrt(strat_counts[k]))
    print(weights.value_counts().describe())
    weights = weights.clip(upper=weights.quantile(0.95))

    # n_groups = df[weight_cols].astype(str).agg('__'.join, axis=1).nunique()
    # print(n_groups, "unique groups")
    # 3. Determine Epoch Size
    # If None, we match the original length.
    # For 1-epoch LoRA, sometimes 1.5x original length is better to see rare samples.
    if total_samples is None:
        total_samples = len(df)

    # 4. Draw the samples
    # replace=True is the 'Upsampling' engine for rare categories
    balanced_df = df.sample(
        n=total_samples,
        replace=True,
        weights=weights,
        random_state=SEED
    )

    # 5. Cleanup
    return balanced_df.drop(columns=['_strat_key']).reset_index(drop=True)

# Generate the balanced training data
train_df_balanced = create_static_stratified_df(
    train_df,
    weight_cols=[ 'answer', 'subject'],
    total_samples=len(train_df) # Or set a custom size like 10000
)

# cols = ["subject", "answer"]
# key = train_df[cols].astype(str).agg("__".join, axis=1)
# print(key.value_counts().describe())

count     14.000000
mean     222.071429
std      296.171589
min        1.000000
25%       18.000000
50%      100.500000
75%      221.750000
max      861.000000
Name: count, dtype: float64


In [ ]:
peda_meta = ["task", "grade_encoded", "subject", "topic", "category", "skill"]

top_n = 15

for i in peda_meta:
  vc = train_df_balanced[i].value_counts(normalize=True) * 100
  if len(vc) > top_n:
    print(f"Top {top_n}")
    print(vc[:top_n], '\n')
  else:
    print("All")
    print(vc, '\n')


All
task
closed choice    99.002895
true-or false     0.997105
Name: proportion, dtype: float64 

All
grade_encoded
7.0     22.450949
5.0     17.497588
6.0     15.825024
3.0     15.632036
4.0     12.158250
2.0     10.582181
1.0      4.631714
11.0     1.029270
0.0      0.192988
Name: proportion, dtype: float64 

All
subject
natural science     58.346735
social science      37.825667
language science     3.827597
Name: proportion, dtype: float64 

All
topic
biology                              21.486008
geography                            19.395304
physics                              15.117401
science-and-engineering-practices     9.617240
us-history                            8.684464
economics                             7.816018
chemistry                             7.044066
earth-science                         4.535220
writing-strategies                    2.894821
world-history                         1.576069
reading-comprehension                 0.932776
literacy-in-science    

In [ ]:
print(train_df_balanced["num_choices"].value_counts(normalize=True))
print(train_df_balanced["answer"].value_counts(normalize=True))
# 1. Answer distribution for Multiple Choice (MC)
# We filter the dataframe where num_choices > 2
for i in range(2, 6):
  print(f"--- Multiple Choice Answer Distribution for Num_choices == {i} ---")
  mc_df = train_df_balanced[train_df_balanced["num_choices"] == i]
  print(mc_df["answer"].value_counts(normalize=True).sort_index() * 100, '\n')



num_choices
3    0.425860
4    0.367964
2    0.163718
5    0.042457
Name: proportion, dtype: float64
answer
0    0.321968
1    0.292699
2    0.257639
3    0.118688
4    0.009006
Name: proportion, dtype: float64
--- Multiple Choice Answer Distribution for Num_choices == 2 ---
answer
0    50.491159
1    49.508841
Name: proportion, dtype: float64 

--- Multiple Choice Answer Distribution for Num_choices == 3 ---
answer
0    32.703927
1    28.851964
2    38.444109
Name: proportion, dtype: float64 

--- Multiple Choice Answer Distribution for Num_choices == 4 ---
answer
0    25.087413
1    22.465035
2    24.562937
3    27.884615
Name: proportion, dtype: float64 

--- Multiple Choice Answer Distribution for Num_choices == 5 ---
answer
0    18.181818
1    14.393939
2     8.333333
3    37.878788
4    21.212121
Name: proportion, dtype: float64 



# Formatting Dataset

In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / "images" / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }



train_ds = ScienceQADataset(train_df_balanced, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

Datasets created: train=3109, val=1048, test=1008


In [ ]:
# Select 150 samples (roughly 10 per topic if you have 14-15 topics)
val_df_sample = val_df.groupby("topic", group_keys=False).apply(
    lambda x: x.sample(min(len(x), len(train_ds) // 10 // 3), random_state=42)
)

# Create the smaller dataset
val_ds_small = ScienceQADataset(val_df_sample, DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Full Validation: {len(val_ds)} | Validation Sample: {len(val_ds_small)}")

Full Validation: 1048 | Validation Sample: 731


/tmp/ipykernel_85136/1144904208.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_df_sample = val_df.groupby("topic", group_keys=False).apply(


# Model Training Setup

### Setting parameters

In [ ]:
import torch
from transformers import AutoTokenizer, AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. 4-bit NF4 Configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load Model and Processor
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


In [ ]:
# target_modules depends on the internal layer names of SmolVLM (usually 'q_proj', 'v_proj')
LORA_RANK = 7
LORA_ALPHA = 2 * LORA_RANK
lora_config = LoraConfig(
      r=LORA_RANK,
      lora_alpha=LORA_ALPHA,
      lora_dropout=0.1,
      target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
      use_dora=True,
      init_lora_weights="gaussian",

  )


model = prepare_model_for_kbit_training(model)
# 3. Apply LoRA
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Check the output: it should be well under 5,000,000

trainable params: 4,490,240 || all params: 511,972,544 || trainable%: 0.8770


### Training collation function

In [ ]:
image_token_id = processor.tokenizer.additional_special_tokens_ids[
            processor.tokenizer.additional_special_tokens.index("<image>")]
def collate_fn(examples):
  texts = []
  images = []
  for example in examples:
      image = example["image"]
      question = example["text"]
      answer = example["answer"]

      if image.mode != 'RGB':
            image = image.convert('RGB')
      if str(answer).strip() != '' or answer is not None:
            image = vision_aug(image)



      messages = [
          {
              "role": "user",
              "content": [
                  {"type": "text", "text": "Answer with the correct choice only."},
                  {"type": "image"},
                  {"type": "text", "text": question}
              ]
          },
          {
              "role": "assistant",
              "content": [
                  {"type": "text", "text": answer}
              ]
          }
      ]
      text = processor.apply_chat_template(messages, add_generation_prompt=False)
      texts.append(text.strip())
      images.append([image])

  batch = processor(text=texts,
                    images=images,
                    return_tensors="pt",
                    padding=True,
                    max_tokens=512,
                    truncation=True,
                    return_token_type_ids=False
                    )

  labels = batch["input_ids"].clone()
  labels[labels == processor.tokenizer.pad_token_id] = -100
  labels[labels == image_token_id] = -100
  batch["labels"] = labels

  return batch

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

OUTPUT_DIR = "./smol-vlm-scienceqa"
# 1. Define Training Args
training_args = TrainingArguments(
    report_to="none",
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4 * 2,
    num_train_epochs=1,
    logging_steps=10,
    # --- STRATEGY 9: GRADIENT CHECKPOINTING ---
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    save_strategy="epoch", # Save manually to avoid filling Kaggle disk
    eval_strategy="epoch",
    fp16=True, # Use bf16 if on A100, otherwise fp16=True for T4
    optim="paged_adamw_8bit", # Saves VRAM
    remove_unused_columns=False,
)

# 2. Data Collator
# This ensures labels are shifted correctly for causal modeling
data_collator = DataCollatorForLanguageModeling(processor.tokenizer, mlm=False)

# 3. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds, # The dataset we built with the oversampler
    eval_dataset = val_ds,
    data_collator=collate_fn,
)

# 4. Start Training
trainer.train()

Keyword argument `max_tokens` is not a valid argument for this processor and will be ignored.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,0.108900,0.274487


TrainOutput(global_step=389, training_loss=0.29297425507886243, metrics={'train_runtime': 14566.0118, 'train_samples_per_second': 0.213, 'train_steps_per_second': 0.027, 'total_flos': 1.2694179844013184e+16, 'train_loss': 0.29297425507886243, 'epoch': 1.0})

In [ ]:
# trainer.train(resume_from_checkpoint=True)

In [ ]:
# Define your path (ideally on Google Drive)
save_directory = "/content/drive/MyDrive/science_qa_model_v5"
SAVE_DIR="/content/drive/MyDrive/science_qa_model_v5"
# 1. Save the LoRA adapters and config
trainer.model.save_pretrained(save_directory)

# 2. Save the processor (tokenizer + image processor)
# This is crucial for your Inference Notebook!
processor.save_pretrained(save_directory)


trainer.model.save_pretrained("./temp_inference_model")
processor.save_pretrained("./temp_inference_model")

['./temp_inference_model/processor_config.json']

# Model Inference

### Clearing GPU cache

In [ ]:
import gc
# 2. Force Python garbage collection
gc.collect()

# 3. Clear the CUDA cache
torch.cuda.empty_cache()

# 4. Optional: Check memory to confirm it's clear
print(f"GPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

GPU Memory Allocated: 621.46 MB
GPU Memory Reserved: 780.00 MB


### Load Saved Model

In [ ]:
# #Run when you want to load a previously trained model
# import torch
# from transformers import AutoModelForVision2Seq, AutoProcessor
# from peft import PeftModel
# SAVE_DIR="/content/drive/MyDrive/science_qa_model_v2"


# # 1. Load the Processor
# processor = AutoProcessor.from_pretrained(SAVE_DIR)

# # 2. Load the Base Model (in 4-bit or 8-bit to save VRAM)
# base_model = AutoModelForVision2Seq.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# # 3. Load your Saved Adapters
# model = PeftModel.from_pretrained(base_model, SAVE_DIR)

# print("Model fully restored and ready for inference!")

### Inference collate function

In [ ]:
def inference_collate_fn(examples):
    texts = []
    images = []
    for example in examples:
        image = example["image"]
        if image.mode != 'RGB':
            image = image.convert('RGB')

        question = example["text"]

        # Inference messages only contain the USER role
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": "Answer with the correct choice only."},
                    {"type": "image"},
                    {"type": "text", "text": question}
                ]
            }
        ]

        # add_generation_prompt=True is the "trigger" for the model to answer
        text = processor.apply_chat_template(messages, add_generation_prompt=True)
        texts.append(text.strip())
        images.append([image])

    # Dynamic padding for efficiency
    batch = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        # max_tokens=512 is safe, but inference usually needs much less
    )

    return batch

### Inference

In [ ]:
from torch.utils.data import DataLoader
from tqdm import tqdm

# 1. Set padding side to left for generation
processor.tokenizer.padding_side = "left"

# 2. Ensure the pad token is correctly identified
processor.tokenizer.pad_token = processor.tokenizer.eos_token

# 1. Initialize the DataLoader with your new inference_collate_fn
# Start with batch_size=4; if VRAM stays low, you can try 8.
test_loader = DataLoader(
    test_ds,
    batch_size=8,
    shuffle=False,
    collate_fn=inference_collate_fn
)

model.eval()
raw_predictions = []

# 2. The Inference Loop
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Predicting Batches"):
        # Move all tensors in the batch to GPU
        inputs = {k: v.to("cuda") if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

        # Generate output
        # max_new_tokens=10 is usually plenty for a single-digit/letter answer
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=10,
            use_cache=True,
            do_sample=False, # Use greedy decoding for consistent results
            pad_token_id=processor.tokenizer.pad_token_id
        )

        # 3. Decode only the NEWLY generated tokens
        # We slice the output to skip the input prompt tokens
        input_len = inputs["input_ids"].shape[1]
        generated_tokens = generated_ids[:, input_len:]

        batch_outputs = processor.batch_decode(generated_tokens, skip_special_tokens=True)
        raw_predictions.extend(batch_outputs)

# 4. Cleanup to release VRAM immediately
del inputs, generated_ids
torch.cuda.empty_cache()

Predicting Batches: 100%|██████████| 126/126 [52:40<00:00, 25.08s/it]


### Submission

In [ ]:
import pandas as pd
import re

# 1. Cleaning function to ensure we get a digit (0-indexed choice)
# def extract_choice(text):
#     # Find the first digit (0, 1, 2, etc.) in the model's output
#     match = re.search(r'\d', text)
#     if match:
#         return match.group(0)
#     return "0" # Default fallback if the model is silent

def extract_choice(text):
  text_strip = text.strip()
  if len(text_strip) <= 0: return 0
  answer = text_strip[0].strip()
  if (answer in ['A', 'B', 'C', 'D']):
    return ord(answer) - ord('A')
  elif (answer in ['0', '1', '2', '3', '4']):
    return int(answer)
  else:
    return 0

# 2. Process the raw results
final_answers = [extract_choice(pred) for pred in raw_predictions]

# 3. Create the submission DataFrame
# Ensure the length of predictions matches your test_df
submission_df = pd.DataFrame({
    "id": test_df["id"],
    "answer": final_answers
})

# Check the first few rows
print(submission_df.head())

# Check the distribution of answers
# If 90% of your answers are '0', something might be wrong with the inference logic!
print("\nAnswer Distribution:")
print(submission_df['answer'].value_counts())

# 4. Save to CSV
submission_path = "submission.csv"
submission_df.to_csv(submission_path, index=False)

print(f"Submission file saved to {submission_path} with {len(submission_df)} rows.")


           id  answer
0  test_01750       3
1  test_00128       1
2  test_02891       3
3  test_02425       3
4  test_00930       1

Answer Distribution:
answer
1    455
0    276
2    179
3     89
4      9
Name: count, dtype: int64
Submission file saved to submission.csv with 1008 rows.


In [ ]:

from google.colab import files

# Replace 'example.csv' with your filename
files.download(submission_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(raw_predictions)
print(final_answers)

[' D', ' 1', ' D', ' 3', ' 1', ' 1', ' D', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', '  D', ' D', ' 3', ' 1', ' 1', ' D', ' D', ' D', ' D', ' D', ' D', ' 1', ' D', ' D', ' D', ' D', ' A', ' 1', ' D', ' 2', ' 1', ' 1', ' 1', ' 1', ' 1', ' 3', ' C', ' 3', ' C', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' D', ' D', ' D', ' 3', ' D', ' D', ' D', ' D', ' D', ' D', ' 1', ' A', ' 1', ' 1', ' 0', ' 0', ' C', ' A', ' B', ' C', ' C', ' A', ' C', ' 1', ' 1', ' 1', ' 1', ' 1', ' B', ' C', ' A', ' B', ' C', ' D', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', ' B', ' 1', ' 1', ' 1', ' 1', ' 1', ' 1', '  B', '  A', ' C', ' A', ' C', ' C', ' C', ' 1', ' 1', ' 1', ' 1', ' 1', ' D', ' 1', ' 1', ' 0', ' 0', ' 1', ' 1', ' B', ' B', ' 1', ' C', ' C', ' 2', ' 1', ' 2', ' 1', ' C', ' 4', ' 1', ' 4', ' 4', ' 4', ' 4', ' 1', ' C', ' 1', ' C', ' 1', ' 1', ' 4', ' C', ' 1', ' 1', ' 1', ' 1', ' 1', ' 0', ' 0', ' 0', ' B', ' 0', ' 0', ' B', ' B', ' 0', ' 0', ' 0', ' B', ' A', 